# 🐺 GökTürk2.5-3B-Thinking — Kaggle T4 · Eğitim → GGUF → Hugging Face

**Başlamadan önce (sağ panel → Settings):**
1. **Accelerator: `GPU T4 x2`** ← *"None"/CPU seçiliyse eğitim yapılamaz; 2. hücre durur ve uyarır.*
2. **Internet: On** (hesabınızda telefon doğrulaması gerekir)
3. **Add-ons → Secrets:** `HF_TOKEN` (**Write** yetkili; huggingface.co → Settings → Access Tokens). İsteğe bağlı: `GEMINI_API_KEY`, `TAVILY_API_KEY`. Her secret'ın yanındaki kutuyu **işaretleyin** (notebook'a bağlanır).

**Sadece GGUF üretmek (eğitim bitti, LoRA hazır):** ayarlarda `LORA_SOURCE`'u doldurun → eğitim atlanır. Bu mod **GPU gerektirmez** (Accelerator: None yeterli, GPU kotası harcanmaz).

**Arka planda çalıştırma:** sağ üst **Save Version → Save & Run All (Commit)**. Hiçbir hücre etkileşim beklemez.

Sonuçta Hugging Face'te 3 depo oluşur: `…/GokTurk2.5-3B-Thinking` (16-bit) · `…-GGUF` (Q4_K_M + Q8_0) · `…-LoRA`

In [ ]:
# ⚙️ AYARLAR — yalnızca burayı düzenleyin
MODEL_SIZE   = "3b"        # "3b" (Q4_K_M ≈ 1.9 GB) · "1.5b" (≈ 1.0 GB, 2× hızlı)
EPOCHS       = 2
MAX_STEPS    = -1          # hızlı deneme: 60 (~5 dk)
QUANTS       = "q4_k_m q8_0"
HF_REPO      = ""          # boş → <hf_kullanıcı_adınız>/GokTurk2.5-3B-Thinking
HF_PRIVATE   = False
USE_DISTILL  = False       # True: GEMINI_API_KEY (+TAVILY_API_KEY) ile ek yüksek kaliteli veri
DISTILL_TOPICS = 300
# ♻️ Eğitimi ATLA, mevcut LoRA'dan GGUF üret:
#   HF deposu : "kullanici_adiniz/GokTurk2.5-3B-Thinking-LoRA"
#   Kaggle    : önceki sürümün çıktısını "Add Input" ile ekleyip klasör yolunu verin,
#               ör. "/kaggle/input/<notebook-adi>/outputs/gokturk/lora_adapter"
LORA_SOURCE  = ""
BRANCH   = "arena/01a0d4d3-gokturk-2-5b-thinking"
REPO_URL = "https://github.com/a23521745-hub/GokTurk-2.5B-Thinking.git"
REPO_DIR = "/kaggle/working/GokTurk"

# 🖥️ GPU KONTROLÜ — CPU'da boşuna saatler harcamamak için en başta
import subprocess, shutil
try:
    gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip()
except Exception:
    gpu = ""
EXPORT_ONLY = bool(LORA_SOURCE.strip())
if EXPORT_ONLY:
    print("♻️ Yalnızca GGUF modu — eğitim atlanacak. GPU:", gpu or "yok (sorun değil, CPU yeterli)")
elif not gpu:
    raise SystemExit("❌ GPU YOK! Settings → Accelerator → 'GPU T4 x2' seçin ve oturumu yeniden başlatın "
                     "(seçenek pasifse haftalık GPU kotanız dolmuştur).")
else:
    print("✅ GPU:", gpu)

In [ ]:
# 📥 Repo
import os
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
for i in range(3):
    if subprocess.call(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR]) == 0:
        break
else:
    raise SystemExit("❌ Repo klonlanamadı (Internet açık mı?)")
os.chdir(REPO_DIR)
print(subprocess.check_output(["git", "log", "--oneline", "-1"], text=True))

In [ ]:
# 📦 Kurulum — sürümler kilitli (datasets<4.4.0: unsloth-zoo gereksinimi; huggingface_hub<1.0) · 3-5 dk
if EXPORT_ONLY:   # birleştirme + dönüştürme: unsloth gerekmez (CPU'da da çalışır)
    !pip install -q peft transformers huggingface_hub sentencepiece protobuf accelerate -c constraints.txt
    !python -c "import peft, transformers, torch, huggingface_hub as h; print('✅ peft', peft.__version__, '| transformers', transformers.__version__, '| hub', h.__version__, '| torch', torch.__version__)"
else:
    !pip install -q -r requirements-train.txt -c constraints.txt
    !pip install -q openai -c constraints.txt
    # Kurulumu AYRI bir süreçte doğrula (notebook çekirdeği torch'u hiç import etmez)
    !python -c "import unsloth, datasets, trl, peft, huggingface_hub as h, torch; print('✅ unsloth', unsloth.__version__, '| datasets', datasets.__version__, '| trl', trl.__version__, '| hub', h.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
# 🔑 Secrets → ortam değişkenleri (alt süreçlere aktarılır; değerler yazdırılmaz)
import sys; sys.path.insert(0, REPO_DIR)
import gokturk_env
status = gokturk_env.load_secrets()
if HF_REPO:
    os.environ["HF_REPO"] = HF_REPO
# Token'ı ŞİMDİ doğrula (eğitimden önce). Geçersizse eğitim yine yapılır, yalnızca yükleme atlanır.
HF_USER = gokturk_env.check_hf_token()
status["HF_TOKEN"] = HF_USER is not None
if not HF_USER:
    print("ℹ️ Eğitim devam edecek; model Output sekmesinden indirilebilir. Token'ı düzeltince yalnızca son hücreyi tekrar çalıştırın.")

In [ ]:
if EXPORT_ONLY:
    print('⏭️ Veri adımı atlandı (yalnızca GGUF modu)')
else:
    # 📚 Veri: (1) /kaggle/input'a eklenmiş train.jsonl  (2) sentetik  (3) isteğe bağlı Gemini+Tavily distill
    import glob
    os.makedirs("data/gokturk6", exist_ok=True)
    attached = sorted(glob.glob("/kaggle/input/**/train.jsonl", recursive=True))
    if attached:
        shutil.copy(attached[0], "data/gokturk6/train.jsonl")
        val = attached[0].replace("train.jsonl", "val.jsonl")
        if os.path.exists(val): shutil.copy(val, "data/gokturk6/val.jsonl")
        print("✓ Ekli veri:", attached[0])
    else:
        parts = ["data/synth6.jsonl"]
        !python prepare_dataset.py synthetic -o data/synth6.jsonl
        parts += sorted(f for f in glob.glob("/kaggle/input/**/*.jsonl", recursive=True) if "seeds" not in f)
        if USE_DISTILL and status["GEMINI_API_KEY"]:
            !python prepare_dataset.py distill --topics {DISTILL_TOPICS} --workers 6 -o data/distill6.jsonl
            if os.path.exists("data/distill6.jsonl"): parts.append("data/distill6.jsonl")
        !python prepare_dataset.py merge {" ".join(parts)} -o data/gokturk6

In [ ]:
# 🚀 Eğitim → LoRA hemen Hugging Face'e (…-LoRA) yüklenir · 3B ≈ 45-70 dk
priv = "--private" if HF_PRIVATE else ""
if EXPORT_ONLY:
    print("⏭️ Eğitim atlandı — LoRA kaynağı:", LORA_SOURCE)
else:
    push = "--push-lora" if status["HF_TOKEN"] else ""
    !python train_unsloth.py --size {MODEL_SIZE} --epochs {EPOCHS} --max-steps {MAX_STEPS} {push} {priv}
    assert os.path.exists("/kaggle/working/outputs/gokturk/lora_adapter/adapter_config.json"), "❌ Eğitim tamamlanmadı — yukarıdaki hatayı inceleyin"

In [ ]:
# 🔗 Merge (CPU, 16-bit) → GGUF (llama.cpp) → Hugging Face  · ~15-25 dk
# Bu hücre hata verirse sadece bu hücreyi yeniden çalıştırmanız yeterli (eğitim tekrarlanmaz).
push = "--push" if status["HF_TOKEN"] else ""
src = ""
if EXPORT_ONLY:
    L = LORA_SOURCE.strip()
    if os.path.isdir(L):
        src = f"--lora {L}"
    else:
        cand = sorted(glob.glob(f"/kaggle/input/**/{L}/adapter_config.json", recursive=True)) if "/" not in L else []
        src = f"--lora {os.path.dirname(cand[0])}" if cand else f"--lora-repo {L}"
    # 16-bit taban modeli açıkça ver (llama.cpp 4-bit bitsandbytes ağırlıkları dönüştüremez)
    src += " --base " + ("Qwen/Qwen2.5-1.5B-Instruct" if MODEL_SIZE == "1.5b" else "Qwen/Qwen2.5-3B-Instruct")
!python push_to_hf.py --quants {QUANTS} {src} {push} {priv}

In [ ]:
# 📦 Çıktılar (Kaggle → Output sekmesinden de indirilebilir)
for root, _, files in os.walk("/kaggle/working/outputs"):
    for f in files:
        p = os.path.join(root, f)
        print(f"{os.path.getsize(p)/1024**2:10.1f} MB  {p}")